# ML Model Monitoring — Walkthrough

This notebook walks through the framework end-to-end:

1. Generate synthetic baseline + current data
2. Construct a `ModelMonitor` for a churn classifier
3. Run drift detection across four scenarios
4. Visualize feature distributions before and after drift
5. Show formatted Slack alert and Snowflake audit payloads (dry run)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from synth_data import generate_reference, generate_current
from monitor import ModelMonitor
from alerting import SlackAlerter, SnowflakeAuditor

## 1. Generate synthetic data

The reference frame is 50K rows drawn from a stable distribution. The current frame is 20K rows, optionally with controlled drift injected.

In [ ]:
reference = generate_reference(n=50_000)
reference.head()

## 2. Configure the monitor

Numeric features get KS + PSI; categorical features get chi-squared. The score column is checked for prediction drift separately.

In [ ]:
monitor = ModelMonitor(
    model_name='churn_classifier_v2',
    reference=reference,
    numeric_features=['monthly_charges', 'tenure_months',
                       'support_tickets_30d', 'bandwidth_gb'],
    categorical_features=['plan_tier', 'payment_method'],
    score_column='churn_score',
)

## 3. Run against four scenarios

Each row of the summary below corresponds to one nightly run.

In [ ]:
scenarios = ['none', 'price_hike', 'ticket_surge', 'plan_mix', 'score_drift']
rows = []
for s in scenarios:
    cur = generate_current(20_000, drift_scenario=s, seed=hash(s) & 0xFFFF)
    report = monitor.run(cur)
    rows.append({
        'scenario': s,
        'severity': report.aggregate_severity,
        'critical_features': ', '.join(report.critical_features()) or '-',
        'warning_features': ', '.join(report.warning_features()) or '-',
    })
pd.DataFrame(rows)

## 4. Inspect drift visually

Confirm that the framework is flagging real distributional differences.

In [ ]:
current_drift = generate_current(20_000, drift_scenario='price_hike', seed=2024)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(reference['monthly_charges'], bins=50, alpha=0.5, label='reference', density=True)
ax[0].hist(current_drift['monthly_charges'], bins=50, alpha=0.5, label='current', density=True)
ax[0].set_title('monthly_charges — price_hike scenario')
ax[0].legend()

current_score = generate_current(20_000, drift_scenario='score_drift', seed=2024)
ax[1].hist(reference['churn_score'], bins=50, alpha=0.5, label='reference', density=True)
ax[1].hist(current_score['churn_score'], bins=50, alpha=0.5, label='current', density=True)
ax[1].set_title('churn_score — score_drift scenario')
ax[1].legend()
plt.tight_layout()

## 5. Alert delivery (dry run)

In production these go to a real Slack webhook and a Snowflake audit table. Dry-run mode returns the payload so you can inspect format.

In [ ]:
report = monitor.run(current_drift)
slack = SlackAlerter(webhook_url='https://example.invalid', dry_run=True)
slack.post(report)

In [ ]:
auditor = SnowflakeAuditor(dry_run=True)
row = auditor.write(report)
{k: (v[:120] + '...' if isinstance(v, str) and len(v) > 120 else v) for k, v in row.items()}